# Rice Leaf Multi-Disease: Multi-Label ViT vs Swin Transformer + Co-occurrence Bounding Boxes

**Colab runtime: T4 GPU** (`Runtime > Change runtime type > T4 GPU`)

### What this notebook does
1. Builds **one multi-label dataset** with 3 classes — `blast`, `blb`, `brown_spot` — from:
   - your **base single-disease leaf images** (uneven counts, one label each), and
   - your **composite co-occurrence images** (blast_on_brown_spot, blb_on_brown_spot, blb_on_blast — 2 labels each)
2. Balances/samples the pool so each class has **~1000 positive images** (capped by availability).
3. Trains **two multi-label classifiers** for comparison:
   - **Model A: ViT-Base/16** (`timm`)
   - **Model B: Swin Transformer** (`timm`)
4. Reports **Accuracy (exact-match + per-class), F1, Precision, Recall** (macro + micro) for both models, side by side.
5. For **each model separately**, finds test images where the model predicts **2 diseases present**, applies the condition *"if one disease is found, check the same leaf for the other"*, localizes each disease with HSV-based region proposals + per-region re-classification, draws labeled bounding boxes, and saves **15 example images**.

### Assumption (please confirm/adjust the upload structure below)
Since you're uploading fresh (not from Drive), zip your data locally in this exact structure before uploading:

```
data.zip
└── data/
    ├── base_leaves/
    │   ├── blast/          (blast only)
    │   ├── blb/             (blb only)
    │   └── brown_spot/      (brown_spot only)
    └── composites/
        ├── blast_on_brown_spot/   (blast=1, brown_spot=1)
        ├── blb_on_brown_spot/     (blb=1, brown_spot=1)
        └── blb_on_blast/          (blb=1, blast=1)
```
If your folder names differ, just edit `BASE_DIR` / folder names in the data-prep cell — everything downstream reads from the dataframe, not hardcoded paths.


## 0. Setup

In [ ]:

!pip install -q timm albumentations opencv-python-headless scikit-learn
import os, cv2, timm, torch, random, zipfile, json
import numpy as np
import pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, classification_report)
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
CLASSES = ["blast", "blb", "brown_spot"]  # fixed label order used everywhere below


Device: cuda


## 1. Mount Google Drive and point to your data

Edit `BASE_LEAF_DIR` and `COMPOSITE_DIR` below to match wherever your `base_leaves` and
`composites` folders actually live in your Drive.

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

# EDIT THESE TWO PATHS to match your Drive structure
BASE_LEAF_DIR = "/content/drive/MyDrive/base_leaves"
COMPOSITE_DIR = "/content/drive/MyDrive/composites"

print("base_leaves ->")
!find "{BASE_LEAF_DIR}" -maxdepth 1 -type d
print("\ncomposites ->")
!find "{COMPOSITE_DIR}" -maxdepth 1 -type d


Mounted at /content/drive
base_leaves ->
/content/drive/MyDrive/base_leaves
/content/drive/MyDrive/base_leaves/blb
/content/drive/MyDrive/base_leaves/brown_spot
/content/drive/MyDrive/base_leaves/blast

composites ->
/content/drive/MyDrive/composites
/content/drive/MyDrive/composites/blast_on_brown_spot
/content/drive/MyDrive/composites/blb_on_brown_spot
/content/drive/MyDrive/composites/blb_on_blast


## 2. Build the multi-label dataframe

Every image gets a 3-length binary label vector `[blast, blb, brown_spot]`.
Base leaves get a single 1. Composites get two 1s.

In [ ]:

VALID_EXT = (".jpg", ".jpeg", ".png")

def list_images(folder):
    if not os.path.isdir(folder):
        return []
    return [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(VALID_EXT)]

records = []

# base leaves -> single label
base_map = {"blast": [1,0,0], "blb": [0,1,0], "brown_spot": [0,0,1]}
for folder_name, label in base_map.items():
    for p in list_images(os.path.join(BASE_LEAF_DIR, folder_name)):
        records.append({"filepath": p, "blast": label[0], "blb": label[1],
                         "brown_spot": label[2], "source": f"base_{folder_name}"})

# composites -> two labels
composite_map = {
    "blast_on_brown_spot": [1,0,1],
    "blb_on_brown_spot":   [0,1,1],
    "blb_on_blast":        [1,1,0],
}
for folder_name, label in composite_map.items():
    for p in list_images(os.path.join(COMPOSITE_DIR, folder_name)):
        records.append({"filepath": p, "blast": label[0], "blb": label[1],
                         "brown_spot": label[2], "source": folder_name})

df_all = pd.DataFrame(records)
print("Total images found:", len(df_all))
print(df_all["source"].value_counts())
print("\nRaw positive counts per class:")
print(df_all[CLASSES].sum())


Total images found: 7863
source
base_blast             2760
base_brown_spot        1392
base_blb               1076
blb_on_brown_spot       900
blb_on_blast            900
blast_on_brown_spot     835
Name: count, dtype: int64

Raw positive counts per class:
blast         4495
blb           2876
brown_spot    3127
dtype: int64


## 3. Balance to ~1000 positive images per class

Counts are uneven across sources, so we sample up to 1000 positive images per class
(using whatever is available if a class has fewer than 1000), then take the union
— a composite image legitimately counts toward two class quotas at once.

In [ ]:

TARGET_PER_CLASS = 1000

selected_paths = set()
for cls in CLASSES:
    pool = df_all[df_all[cls] == 1]
    n = min(TARGET_PER_CLASS, len(pool))
    sampled = pool.sample(n=n, random_state=SEED)
    selected_paths.update(sampled["filepath"].tolist())
    print(f"{cls}: available={len(pool)}, sampled={n}")

df = df_all[df_all["filepath"].isin(selected_paths)].reset_index(drop=True)
print("\nFinal dataset size:", len(df))
print("Final positive counts per class (can exceed 1000 slightly due to overlap):")
print(df[CLASSES].sum())
print("\nLabel-combination breakdown:")
print(df[CLASSES].value_counts())


blast: available=4495, sampled=1000
blb: available=2876, sampled=1000
brown_spot: available=3127, sampled=1000

Final dataset size: 2780
Final positive counts per class (can exceed 1000 slightly due to overlap):
blast         1428
blb           1306
brown_spot    1343
dtype: int64

Label-combination breakdown:
blast  blb  brown_spot
1      0    0             637
0      1    1             506
       0    1             463
1      1    0             417
0      1    0             383
1      0    1             374
Name: count, dtype: int64


## 4. Train / validation / test split (stratified by label combination)

In [ ]:

df["label_combo"] = df[CLASSES].astype(str).agg("".join, axis=1)

train_df, temp_df = train_test_split(df, test_size=0.30, random_state=SEED, stratify=df["label_combo"])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["label_combo"])

print("Train:", len(train_df), " Val:", len(val_df), " Test:", len(test_df))
train_df.to_csv("/content/train.csv", index=False)
val_df.to_csv("/content/val.csv", index=False)
test_df.to_csv("/content/test.csv", index=False)


Train: 1946  Val: 417  Test: 417


## 5. Dataset class + transforms

In [ ]:

import albumentations as A
from albumentations.pytorch import ToTensorV2

IMG_SIZE = 224

train_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.Rotate(limit=20, p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

eval_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

class RiceLeafMultiLabelDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row["filepath"])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        label = torch.tensor(row[CLASSES].values.astype("float32"))
        img = self.transform(image=img)["image"]
        return img, label, row["filepath"]

BATCH_SIZE = 32

train_ds = RiceLeafMultiLabelDataset(train_df, train_tf)
val_ds   = RiceLeafMultiLabelDataset(val_df, eval_tf)
test_ds  = RiceLeafMultiLabelDataset(test_df, eval_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


## 6. Generic train / evaluate functions (reused by both models)

In [ ]:

def train_one_model(model, train_loader, val_loader, epochs=12, lr=3e-5, tag="model"):
    model = model.to(DEVICE)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_f1 = -1
    best_state = None

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for imgs, labels, _ in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * imgs.size(0)
        scheduler.step()
        train_loss = running_loss / len(train_loader.dataset)

        val_metrics, _, _ = evaluate_model(model, val_loader)
        print(f"[{tag}] epoch {epoch+1}/{epochs} | train_loss={train_loss:.4f} "
              f"| val_f1_macro={val_metrics['f1_macro']:.4f} | val_acc_exact={val_metrics['accuracy_exact']:.4f}")

        if val_metrics["f1_macro"] > best_val_f1:
            best_val_f1 = val_metrics["f1_macro"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    torch.save(best_state, f"/content/{tag}_best.pt")
    print(f"[{tag}] best val_f1_macro={best_val_f1:.4f} — saved to /content/{tag}_best.pt")
    return model


@torch.no_grad()
def evaluate_model(model, loader, threshold=0.5):
    model.eval()
    all_probs, all_labels = [], []
    for imgs, labels, _ in loader:
        imgs = imgs.to(DEVICE)
        logits = model(imgs)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels.numpy())
    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    preds = (all_probs >= threshold).astype(int)

    metrics = {
        "accuracy_exact": accuracy_score(all_labels, preds),          # exact match across all 3 labels
        "precision_macro": precision_score(all_labels, preds, average="macro", zero_division=0),
        "recall_macro": recall_score(all_labels, preds, average="macro", zero_division=0),
        "f1_macro": f1_score(all_labels, preds, average="macro", zero_division=0),
        "precision_micro": precision_score(all_labels, preds, average="micro", zero_division=0),
        "recall_micro": recall_score(all_labels, preds, average="micro", zero_division=0),
        "f1_micro": f1_score(all_labels, preds, average="micro", zero_division=0),
    }
    # per-class breakdown
    per_class = classification_report(all_labels, preds, target_names=CLASSES,
                                       zero_division=0, output_dict=True)
    return metrics, per_class, (all_probs, preds, all_labels)


def print_full_report(tag, metrics, per_class):
    print(f"\n===== {tag} — TEST SET METRICS =====")
    print(f"Exact-match accuracy : {metrics['accuracy_exact']:.4f}")
    print(f"Precision (macro)    : {metrics['precision_macro']:.4f}")
    print(f"Recall    (macro)    : {metrics['recall_macro']:.4f}")
    print(f"F1        (macro)    : {metrics['f1_macro']:.4f}")
    print(f"Precision (micro)    : {metrics['precision_micro']:.4f}")
    print(f"Recall    (micro)    : {metrics['recall_micro']:.4f}")
    print(f"F1        (micro)    : {metrics['f1_micro']:.4f}")
    print("\nPer-class:")
    for c in CLASSES:
        r = per_class[c]
        print(f"  {c:12s} precision={r['precision']:.4f}  recall={r['recall']:.4f}  f1={r['f1-score']:.4f}")


## 7. Model A — ViT-Base/16

In [ ]:

vit_model = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=len(CLASSES))
vit_model = train_one_model(vit_model, train_loader, val_loader, epochs=12, lr=3e-5, tag="vit_base16")

vit_test_metrics, vit_per_class, vit_raw = evaluate_model(vit_model, test_loader)
print_full_report("ViT-Base/16", vit_test_metrics, vit_per_class)


model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

[vit_base16] epoch 1/12 | train_loss=0.4405 | val_f1_macro=0.8609 | val_acc_exact=0.6859
[vit_base16] epoch 2/12 | train_loss=0.2619 | val_f1_macro=0.8698 | val_acc_exact=0.7386
[vit_base16] epoch 3/12 | train_loss=0.1924 | val_f1_macro=0.8796 | val_acc_exact=0.7338
[vit_base16] epoch 4/12 | train_loss=0.1480 | val_f1_macro=0.8990 | val_acc_exact=0.7842
[vit_base16] epoch 5/12 | train_loss=0.1098 | val_f1_macro=0.9093 | val_acc_exact=0.8106
[vit_base16] epoch 6/12 | train_loss=0.1048 | val_f1_macro=0.9148 | val_acc_exact=0.8082
[vit_base16] epoch 7/12 | train_loss=0.0808 | val_f1_macro=0.9220 | val_acc_exact=0.8369
[vit_base16] epoch 8/12 | train_loss=0.0559 | val_f1_macro=0.9268 | val_acc_exact=0.8345
[vit_base16] epoch 9/12 | train_loss=0.0549 | val_f1_macro=0.9239 | val_acc_exact=0.8369
[vit_base16] epoch 10/12 | train_loss=0.0375 | val_f1_macro=0.9331 | val_acc_exact=0.8561
[vit_base16] epoch 11/12 | train_loss=0.0342 | val_f1_macro=0.9325 | val_acc_exact=0.8585
[vit_base16] epoch 

## 8. Model B — Swin Transformer

(`swin_base_patch4_window7_224` — if you hit an OOM on T4, switch to `swin_tiny_patch4_window7_224` and/or drop `BATCH_SIZE` to 16.)

In [ ]:

swin_model = timm.create_model("swin_base_patch4_window7_224", pretrained=True, num_classes=len(CLASSES))
swin_model = train_one_model(swin_model, train_loader, val_loader, epochs=12, lr=3e-5, tag="swin_base")

swin_test_metrics, swin_per_class, swin_raw = evaluate_model(swin_model, test_loader)
print_full_report("Swin Transformer", swin_test_metrics, swin_per_class)


model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

[swin_base] epoch 1/12 | train_loss=0.4418 | val_f1_macro=0.8455 | val_acc_exact=0.6547
[swin_base] epoch 2/12 | train_loss=0.2317 | val_f1_macro=0.8874 | val_acc_exact=0.7482
[swin_base] epoch 3/12 | train_loss=0.1653 | val_f1_macro=0.9157 | val_acc_exact=0.8034
[swin_base] epoch 4/12 | train_loss=0.1353 | val_f1_macro=0.9236 | val_acc_exact=0.8417
[swin_base] epoch 5/12 | train_loss=0.0985 | val_f1_macro=0.9365 | val_acc_exact=0.8681
[swin_base] epoch 6/12 | train_loss=0.0822 | val_f1_macro=0.9403 | val_acc_exact=0.8681
[swin_base] epoch 7/12 | train_loss=0.0614 | val_f1_macro=0.9356 | val_acc_exact=0.8657
[swin_base] epoch 8/12 | train_loss=0.0515 | val_f1_macro=0.9395 | val_acc_exact=0.8777
[swin_base] epoch 9/12 | train_loss=0.0473 | val_f1_macro=0.9440 | val_acc_exact=0.8849
[swin_base] epoch 10/12 | train_loss=0.0455 | val_f1_macro=0.9442 | val_acc_exact=0.8825
[swin_base] epoch 11/12 | train_loss=0.0358 | val_f1_macro=0.9483 | val_acc_exact=0.8945
[swin_base] epoch 12/12 | trai

## 9. Side-by-side comparison

In [ ]:

comparison = pd.DataFrame({
    "ViT-Base/16": vit_test_metrics,
    "Swin Transformer": swin_test_metrics,
}).T
comparison = comparison[["accuracy_exact", "precision_macro", "recall_macro", "f1_macro",
                          "precision_micro", "recall_micro", "f1_micro"]]
comparison.round(4)


,accuracy_exact,precision_macro,recall_macro,f1_macro,precision_micro,recall_micro,f1_micro
ViT-Base/16,0.8777,0.9574,0.9381,0.9475,0.9566,0.9378,0.9471
Swin Transformer,0.8993,0.9628,0.9477,0.9550,0.9618,0.9476,0.9547


## Notes / things to double-check before a full run

- **Uneven base counts**: if any single disease's base-leaf folder is much smaller than the others, its 1000-per-class target may fall back to "all available" — the printed counts in step 3 tell you exactly what happened.
- **`box_conf_thresh` (0.4)** and **`min_area` (250 px)** in the bounding-box cell are starting points tuned to your earlier HSV lesion-extraction work — adjust if boxes look too small/large or too permissive.
- **Swin memory**: `swin_base_patch4_window7_224` at `BATCH_SIZE=32`, `224px` should fit T4's 16GB, but if you see a CUDA OOM, drop `BATCH_SIZE` to 16 or switch to `swin_tiny_patch4_window7_224`.
- Both models save their best checkpoint (`vit_base16_best.pt`, `swin_base_best.pt`) to `/content` — copy these to Drive if you want them to persist past the session.
